> 在线讲义：[第 19 章 · 过估计与 Double DQN：max 会挑到"运气好"的那个](https://wikieden.github.io/rl-from-scratch/lessons/19-double-dqn.html) · 本书《从零学强化学习》

# 第 19 章 · 过估计与 Double DQN：max 会挑到"运气好"的那个

本章回答：Q-learning 和 DQN 学到的 Q 值为什么普遍偏高？"挑最大"这个动作本身有什么问题？用两个估计怎么修？

配套讲义：仓库里 `cs285-deep-rl/notes/lessons/19-double-dqn.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)  # 打印保留 3 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 代码 | 跑出来 |
|---|---|---|
| 10 个动作重复 2 万次 | `max_bias()` | 单估计 0.691，双估计 −0.000 |
| 手算一步 | `targets(0.0, 0.9, [1.0, 2.0, 0.5], [1.2, 0.7, 1.5])` | DQN 1.35，Double DQN 0.63 |

## 1. 先把要用的东西定义好

这几格只是定义（环境、函数），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `max_bias`**（来自 `ch19_double_dqn.py`）：每个动作真实价值都是 0，估计 = n_samples 次带噪声（标准差 1）的平均。

In [ ]:
def max_bias(n_actions=10, n_samples=5, trials=20000, seed=0):
    """每个动作真实价值都是 0，估计 = n_samples 次带噪声（标准差 1）的平均。"""
    rng = np.random.default_rng(seed)
    single, double = [], []
    for _ in range(trials):
        Q1 = rng.normal(0, 1, (n_actions, n_samples)).mean(axis=1)
        Q2 = rng.normal(0, 1, (n_actions, n_samples)).mean(axis=1)
        single.append(Q1.max())  # 同一组估计既挑又打分
        double.append(Q2[np.argmax(Q1)])  # Q1 挑动作，Q2 打分
    return float(np.mean(single)), float(np.mean(double))

**函数 `targets`**（来自 `ch19_double_dqn.py`）：DQN 目标用目标网络自己挑 + 自己打分；Double DQN 用在线网络挑、目标网络打分。

In [ ]:
def targets(r, gamma, q_online_next, q_target_next):
    """DQN 目标用目标网络自己挑 + 自己打分；Double DQN 用在线网络挑、目标网络打分。"""
    dqn = r + gamma * q_target_next.max()
    a = int(np.argmax(q_online_next))
    ddqn = r + gamma * q_target_next[a]
    return dqn, ddqn

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
s, d = max_bias()

In [ ]:
print(f"10 个动作真值都是 0：单估计 E[max] = {s:.3f}，双估计 = {d:.3f}")

In [ ]:
dqn, ddqn = targets(0.0, 0.9, np.array([1.0, 2.0, 0.5]), np.array([1.2, 0.7, 1.5]))

In [ ]:
print(f"手算一步：DQN 目标 = {dqn:.2f}，Double DQN 目标 = {ddqn:.2f}")

## 3. 代码里才看得到的

在第 18 章的 DQN 代码里改成 Double DQN，只要把 `target(NS).max(dim=1).values` 换成 `target(NS).gather(1, q(NS).argmax(dim=1, keepdim=True)).squeeze(1)`：用主网络 `q` 的 argmax 去目标网络 `target` 里取值。

## 4. 自己改一改

讲义里建议的几个小实验（数字是实跑出来的，改完对一对）：

1. 换动作个数：`max_bias(n_actions=2 / 10 / 100)` → 单估计偏 0.254、0.691、1.121，双估计都在 0 附近。动作越多，挑出来的最大值偏得越高。
2. 换每个估计用的样本数：`n_samples=1` → 偏 1.546；`n_samples=50` → 偏 0.217。估计越准，偏差越小，但不会消失。

讲义末尾的练习题也可以在下面的空格子里用代码验算：先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 十个一样好，max 挑到运气最好的那个，估计就偏高；
> 目标里有 max，偏高一步步往前传；
> 挑和打分分开：一组挑，另一组打分，噪声不相关就不偏；
> Double DQN：主网络挑，目标网络打分，只改一行。